# Optimised Solution

## Possibly necessary OS settings

In [ ]:
import os
import sys

os.environ["JAVA_HOME"] = "Insert Java openJDK 17 path here"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

python_exec = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exec
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exec

### 1. Load and clean the data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

In [ ]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

import csv

spark = SparkSession.builder.master("local[*]").appName("clean-us-accidents").config("spark.driver.memory", "8g").getOrCreate()
sc = spark.sparkContext

lines = sc.textFile("US_Accidents_March23.csv", minPartitions=8)
header = lines.first()

def parse_csv_line(line: str):
    return next(csv.reader([line]))

header_cols = parse_csv_line(header)
rows_rdd = (
    lines.filter(lambda x: x != header)
         .map(parse_csv_line)
         .map(lambda vals: dict(zip(header_cols, vals)))
).persist(StorageLevel.MEMORY_AND_DISK)


In [ ]:
import math
import csv
from typing import Dict, Any, Optional, List, Iterable, Tuple
from pyspark import RDD

US_STATES = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA",
    "ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK",
    "OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
}

WEATHER_BOUNDS = {
    "Temperature(F)": (-70.0, 130.0),
    "Wind_Chill(F)": (-50.0, 130.0),
    "Humidity(%)": (0.0, 100.0),
    "Pressure(in)": (15.0, 32.0),
    "Visibility(mi)": (0.0, 100.0),
    "Wind_Speed(mph)": (0.0, 200.0),
    "Precipitation(in)": (0.0, 50.0),
}

COLS_TO_DROP = {"ID", "Source", "Zipcode", "Timezone", "Airport_Code", "End_Lat", "End_Lng"}

WIND_DIR_MAP = {
    "VARIABLE": "VAR",
    "VAR": "VAR",
    "CALM": "CALM",
    "NORTH": "N",
    "SOUTH": "S",
    "EAST": "E",
    "WEST": "W"
}

TWILIGHT_COLS = (
    "Sunrise_Sunset",
    "Civil_Twilight",
    "Nautical_Twilight",
    "Astronomical_Twilight"
)

BOOL_COLS = (
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
    "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop"
)

def _safe_float(x: Any) -> Optional[float]:
    try:
        if x is None:
            return None
        if isinstance(x, str):
            x = x.strip()
            if x == "":
                return None
        val = float(x)
        if math.isnan(val):
            return None
        return val
    except (TypeError, ValueError):
        return None

def _safe_int_from_float(x: Any) -> Optional[int]:
    if x is None:
        return None
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return None
    return int(round(float(x)))

# Assume missing boolean values to be false
def _normalize_bool(v: Any) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, str):
        s = v.strip().lower()
        return s == "true"
    return False

def _compute_median(vals: List[float]) -> Optional[float]:
    if not vals:
        return None
    vals.sort()
    n = len(vals)
    mid = n // 2
    if n % 2 == 1:
        return float(vals[mid])
    return (vals[mid - 1] + vals[mid]) / 2.0

def _clean_row(
    row: Dict[str, Any],
    weather_cols: List[str]
) -> Optional[Dict[str, Any]]:
    row = dict(row)

    # Trim strings
    for k, v in row.items():
        if isinstance(v, str):
            row[k] = v.strip()

    # Must have Start_Time
    st = row.get("Start_Time")
    if st is None or (isinstance(st, str) and st.strip() == ""):
        return None

    # Severity in [1, 4]
    if "Severity" in row:
        sev = _safe_int_from_float(row.get("Severity"))
        if sev is None or not (1 <= sev <= 4):
            return None
        row["Severity"] = sev

    # Coordinates
    lat = _safe_float(row.get("Start_Lat"))
    if lat is None or not (-90.0 <= lat <= 90.0):
        return None
    row["Start_Lat"] = lat

    lng = _safe_float(row.get("Start_Lng"))
    if lng is None or not (-180.0 <= lng <= 180.0):
        return None
    row["Start_Lng"] = lng

    # State
    if "State" in row:
        state = row.get("State")
        if isinstance(state, str) and state.strip():
            state = state.upper()
            row["State"] = state if state in US_STATES else None
        else:
            row["State"] = None

    # Weather bounds
    for c in weather_cols:
        lo, hi = WEATHER_BOUNDS[c]
        val = _safe_float(row.get(c))
        if val is None:
            row[c] = None
        else:
            row[c] = val if lo <= val <= hi else None

    # Wind direction
    if "Wind_Direction" in row:
        wd = row.get("Wind_Direction")
        if isinstance(wd, str) and wd.strip():
            wd_up = wd.upper()
            row["Wind_Direction"] = WIND_DIR_MAP.get(wd_up, wd_up)
        else:
            row["Wind_Direction"] = None

    # Twilight columns
    for c in TWILIGHT_COLS:
        if c in row:
            val = row.get(c)
            if isinstance(val, str) and val.strip():
                val_t = val.title()
                row[c] = val_t if val_t in ("Day", "Night") else "Unknown"
            else:
                row[c] = "Unknown"

    # Boolean columns
    for c in BOOL_COLS:
        if c in row:
            row[c] = _normalize_bool(row.get(c))

    # Drop redundant columns
    for c in COLS_TO_DROP:
        row.pop(c, None)

    return row

def clean_us_accidents_pyspark(
    rows: RDD[Dict[str, Any]],
    header_cols: List[str],
    sample_fraction_for_median: float = 0.01,
    max_sample_rows: int = 50000,
) -> RDD[dict[str, Any | None]]:
    sc = rows.context
    weather_cols = [c for c in WEATHER_BOUNDS if c in header_cols]
    output_cols = [c for c in header_cols if c not in COLS_TO_DROP]

    def key_by_id(iterator: Iterable[Dict[str, Any]]) -> Iterable[Tuple[str, Dict[str, Any]]]:
        for row in iterator:
            rid = str(row.get("ID", "") or "").strip()
            if rid:
                yield (rid, row)

    deduped_rows = (
        rows
        .mapPartitions(key_by_id)
        .reduceByKey(lambda a, b: a)
        .values()
    )

    def clean_partition(iterator: Iterable[Dict[str, Any]]) -> Iterable[Dict[str, Any]]:
        local_weather_cols = weather_cols
        for row in iterator:
            cleaned = _clean_row(row, local_weather_cols)
            if cleaned is not None:
                yield cleaned

    cleaned = deduped_rows.mapPartitions(clean_partition)

    medians: Dict[str, Optional[float]] = {}

    if weather_cols:
        # Compute median using a sample of the data
        sampled_weather_rows = (
            cleaned
            .sample(False, sample_fraction_for_median, seed=1)
            .map(lambda r: tuple(r.get(c) for c in weather_cols))
            .take(max_sample_rows)
        )

        for i, c in enumerate(weather_cols):
            vals = []
            for row_vals in sampled_weather_rows:
                v = row_vals[i]
                if v is not None and not math.isnan(v):
                    vals.append(float(v))
            medians[c] = _compute_median(vals)

    medians_bc = sc.broadcast(medians)

    def fill_partition(iterator):
        local_medians = medians_bc.value
        for r in iterator:
            r = dict(r)
            for c in weather_cols:
                if r.get(c) is None and local_medians.get(c) is not None:
                    r[c] = float(local_medians[c])
            yield {col: r.get(col) for col in output_cols}
    final_rdd = cleaned.mapPartitions(fill_partition)
    return final_rdd

In [ ]:
rdd_cleaned = clean_us_accidents_pyspark(
    rows=rows_rdd,
    header_cols=header_cols,
)
parsed_rdd = rdd_cleaned.persist(StorageLevel.MEMORY_AND_DISK)
parsed_rdd.count()
weather_rdd = parsed_rdd
poi_rdd = parsed_rdd
time_study_rdd = parsed_rdd
location_rdd = parsed_rdd

### 2. Analysis of Time of the accident

In [ ]:
# Collect hour_counts
hour_counts = (
    time_study_rdd
    .map(lambda row: (int(row['Start_Time'][11:13]), 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

# Sort by hour
hour_counts = sorted(hour_counts, key =lambda x: x[0])

# Split into lists
hours = [x [0] for x in hour_counts]
counts = [x [1] for x in hour_counts]

# Plot
plt.figure(figsize =( 15,10))
plt.bar(hours, counts)
plt.xticks(range(24))
plt.grid(axis = 'y')
plt.xlabel("Hour of Day")
plt.ylabel("Number of Occurrence")
plt.title("Accidents Count By Time of Day")
plt.show()

In [ ]:
# Collect year_counts
year_counts = (
    time_study_rdd
    .map(lambda row: (datetime.fromisoformat(row['Start_Time']).year, 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

# Sort by year
year_counts = sorted(year_counts, key = lambda x: x[0])

# Split into lists
years = [x[0] for x in year_counts]
counts = [x[1] for x in year_counts]

# Plot
plt.figure(figsize=(15,10))
plt.barh(years, counts)

plt.grid(axis='x')
plt.xlabel("Number of Occurrence")
plt.ylabel("Year")
plt.title("Accidents Distribution by Year")

plt.ticklabel_format(style = 'plain', axis = 'x')

plt.show()

### 3. Analysis of Points of Interest at location of Accident

In [ ]:
poi_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "No_Exit",
    "Roundabout", "Station", "Stop", "Traffic_Calming",
    "Traffic_Signal", "Turning_Loop"
]

def extract_poi_counts(row):
    for col in poi_cols:
        val = row.get(col)
        if val is True:
            yield (col, 1)
        elif isinstance(val, str) and val.strip().lower() == "true":
            yield (col, 1)

poi_counts_rdd = (
    poi_rdd
    .flatMap(extract_poi_counts)
    .reduceByKey(lambda a, b: a + b)
)

counts_list = poi_counts_rdd.collect()

counts_dict = {col: 0 for col in poi_cols}
counts_dict.update(dict(counts_list))

sorted_counts = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)

labels = [x[0] for x in sorted_counts]
counts = [x[1] for x in sorted_counts]

plt.figure(figsize=(20, 5))
plt.bar(labels, counts, color="coral", width=0.9)
plt.ylim((0, 1300000))
plt.xlabel("POI", fontsize=13)
plt.ylabel("Number of Accidents", fontsize=13)
plt.title("Accident by proximity to POI", fontsize=13)
plt.xticks(rotation=0, fontsize=13)
plt.tight_layout()
plt.show()

We see then that the majority of accidents take place at junctions. This is indicated by Traffic_signal, Crossing, and Stop. With all other POIs contributing a small number. We will now look at the top 5 POI features by severity to see for any patterns.

In [ ]:
colors = ['#fee5d9', '#fcae91', '#fb6a4a', '#de2d26', '#a50f15']

severity_cols = ["Traffic_Signal", "Crossing", "Stop", "Station", "Amenity"]

def extract_severity_poi_pairs(row):
    sev = row.get("Severity")
    if sev is None:
        return []

    out = []
    for col in severity_cols:
        if row.get(col) is True:
            out.append(((sev, col), 1))
    return out

severity_poi_counts = (
    poi_rdd
    .flatMap(extract_severity_poi_pairs)
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

counts_dict = {(sev, poi): count for (sev, poi), count in severity_poi_counts}

severity_levels = sorted(
    {sev for sev, _ in counts_dict.keys()},
    key=int
)

plot_rows = []
for poi in severity_cols:
    row = []
    for sev in severity_levels:
        row.append(counts_dict.get((sev, poi), 0))
    plot_rows.append(row)

plot_data = pd.DataFrame(
    plot_rows,
    index=severity_cols,
    columns=severity_levels
)

ax = plot_data.plot(kind="bar", figsize=(10, 6), color=colors)
ax.set_xlabel("POI")
ax.set_ylabel("Count of True values")
ax.set_title("Top 5 POIs by Severity")
plt.xticks(rotation=0)
plt.legend('', frameon=False)
plt.tight_layout()
plt.show()

### 4. Weather Analysis

In [ ]:
header_list = header_cols

def get_column_processor(idx):
    def process_partition(iterator):
        for row in iterator:
            try:
                val = row[idx]
                if val and val.strip():
                    yield float(val)
            except (ValueError, IndexError):
                continue
    return process_partition

In [ ]:
def plot_histogram(col):
    col_rdd = (
        weather_rdd
        .map(lambda row: row.get(col))
        .filter(lambda x: x is not None)
        .map(float)
    )

    num_bins = 30
    min_col = col_rdd.min()
    max_col = col_rdd.max()

    if min_col == max_col:
        print(f"Column {col} has only one value: {min_col}")
        return

    width = (max_col - min_col) / num_bins

    def assign_to_bin(value):
        bin_idx = int((value - min_col) / width)
        if bin_idx >= num_bins:
            bin_idx = num_bins - 1
        return (bin_idx, 1)

    bin_counts_rdd = (
        col_rdd
        .map(assign_to_bin)
        .reduceByKey(lambda a, b: a + b)
        .sortByKey()
    )

    bin_counts = bin_counts_rdd.collect()

    results_dict = dict(bin_counts)
    final_counts = [results_dict.get(i, 0) for i in range(num_bins)]
    final_edges = [min_col + (i * width) for i in range(num_bins)]

    plt.figure(figsize=(8, 5))
    plt.bar(final_edges, final_counts, width=width, align="edge")
    plt.xlabel(col)
    plt.ylabel("Number of Accidents")
    plt.title(f"Distribution of Accidents by {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_histogram("Temperature(F)")

In [ ]:
plot_histogram("Wind_Chill(F)")

In [ ]:
plot_histogram("Humidity(%)")

In [ ]:
plot_histogram("Pressure(in)")

In [ ]:
plot_histogram("Visibility(mi)")

In [ ]:
def get_wind_dir(iterator):
    for row in iterator:
        try:
            val = row.get("Wind_Direction")
            if val is not None:
                val = str(val).strip()
                if val:
                    yield (val, 1)
        except AttributeError:
            continue

wind_direction_counts = (
    weather_rdd
    .mapPartitions(get_wind_dir)
    .reduceByKey(lambda a, b: a + b)
)

sorted_results = (
    wind_direction_counts
    .sortBy(lambda x: x[1], ascending=True)
    .collect()
)

if sorted_results:
    labels, counts = zip(*sorted_results)

    plt.figure(figsize=(10, 8))
    plt.barh(labels, counts)
    plt.title("Wind Direction Distribution", fontsize=14)
    plt.xlabel("Count", fontsize=12)
    plt.ylabel("Wind Direction", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No wind direction data available.")

In [ ]:
plot_histogram("Wind_Speed(mph)")

In [ ]:
plot_histogram("Precipitation(in)")

In [ ]:
detailed_categories = {
    'Rainy': ['rain', 'rainy', 'drizzle', 'shower', 'raining', 'rainfall', 'precipitation'],
    'Cloudy': ['cloud', 'cloudy', 'overcast', 'mostly cloudy', 'partly cloudy'],
    'Clear': ['clear', 'sunny', 'fair'],
    'Snowy': ['snow', 'snowy', 'blizzard', 'sleet', 'ice', 'wintry', 'hail'],
    'Foggy': ['fog', 'foggy', 'mist', 'haze', 'smoke', 'dust', 'ash', 'sand'],
    'Windy': ['wind', 'windy', 'breezy'],
    'Storm': ['storm', 'thunder', 'lightning', 'squall', 'tornado']
}

def detailed_categorize_rdd(row):
    condition = row.get("Weather_Condition")

    if condition is None or str(condition).strip() == "":
        return "Unknown"

    condition_lower = str(condition).lower()

    for category, keywords in detailed_categories.items():
        if any(keyword in condition_lower for keyword in keywords):
            return category

    return "Other"

category_pairs = weather_rdd.map(lambda row: (detailed_categorize_rdd(row), 1))
weather_counts_rdd = category_pairs.reduceByKey(lambda a, b: a + b)

final_results = weather_counts_rdd.collect()
final_results.sort(key=lambda x: x[1])

if final_results:
    categories = [x[0] for x in final_results]
    counts = [x[1] for x in final_results]

    plt.figure(figsize=(10, 8))
    plt.barh(categories, counts)
    plt.title("Weather Category Distribution", fontsize=14)
    plt.xlabel("Count", fontsize=12)
    plt.ylabel("Weather Category", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No weather category data available.")

In [ ]:
def get_severity_weather_pairs(var_name):
    def process_partition(iterator):
        for row in iterator:
            try:
                sev = row.get("Severity")
                val = row.get(var_name)

                if sev is None or val is None:
                    continue

                yield (sev, float(val))
            except (ValueError, TypeError, AttributeError):
                continue

    return process_partition

weather_cols = ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)']
plot_data = {}

for var in weather_cols:
    grouped_rdd = (
        weather_rdd
        .mapPartitions(get_severity_weather_pairs(var))
        .groupByKey()
    )

    stats_by_severity = {}
    for severity, values in grouped_rdd.collect():
        sorted_vals = sorted(list(values))
        if not sorted_vals:
            continue

        n = len(sorted_vals)
        stats_by_severity[severity] = {
            'min': sorted_vals[0],
            'q1': sorted_vals[int(n * 0.25)],
            'med': sorted_vals[int(n * 0.5)],
            'q3': sorted_vals[int(n * 0.75)],
            'max': sorted_vals[-1]
        }

    plot_data[var] = stats_by_severity

# Plotting
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, var in zip(axes.flatten(), weather_cols):
    sev_levels = sorted(plot_data[var].keys())
    box_stats = [
        {
            "label": s,
            "whislo": plot_data[var][s]['min'],
            "q1": plot_data[var][s]['q1'],
            "med": plot_data[var][s]['med'],
            "q3": plot_data[var][s]['q3'],
            "whishi": plot_data[var][s]['max']
        }
        for s in sev_levels
    ]
    ax.bxp(box_stats, showfliers=False)
    ax.set_title(f'{var} by Severity')

plt.tight_layout()
plt.show()

### 5. Create a Map Distribution of the locations for each Accident

In [ ]:
# Use location_rdd
import time

# Start measuring runtime
start_time = time.time()

# Keep only rows that have latitude and longitude values
location_pairs = (
    location_rdd
    .filter(lambda row: row.get("Start_Lat") and row.get("Start_Lng"))

# Round coordinates so nearby accidents fall into the same grid
    .map(lambda row: (
        (
            round(float(row.get("Start_Lat")), 1),
            round(float(row.get("Start_Lng")), 1)
        ),
        1
    ))
)

# Count accidents in each rounded coordinate grid
location_counts = location_pairs.reduceByKey(lambda a, b: a + b)

# Get top 10 accident hotspot grids
top_10_hotspots = location_counts.takeOrdered(10, key=lambda x: -x[1])

# Stop measuring runtime
end_time = time.time()

# Print results
print("Top 10 accident hotspot grids:")
for hotspot in top_10_hotspots:
    print(hotspot)

print("Location analysis runtime:", end_time - start_time, "seconds")

In [ ]:
import pandas as pd
import plotly.express as px
# Start measuring runtime
start_time = time.time()
# Use more than 10 so the map is not empty
top_hotspots = location_counts.takeOrdered(30, key=lambda x: -x[1])

# Convert to pandas dataframe
df_hotspots = pd.DataFrame(top_hotspots, columns=["coords", "count"])
df_hotspots["lat"] = df_hotspots["coords"].apply(lambda x: x[0])
df_hotspots["lng"] = df_hotspots["coords"].apply(lambda x: x[1])

print(df_hotspots)

fig = px.scatter_mapbox(
    df_hotspots,
    lat="lat",
    lon="lng",
    size="count",
    color="count",
    hover_data=["lat", "lng", "count"],
    zoom=3,
    center={"lat": 39.5, "lon": -98.35},
    mapbox_style="carto-positron",
    title="Top Accident Hotspot Grids in the US",
    size_max=30
)

fig.update_layout(height=650)
fig.show()

end_time = time.time()
print("Visualization runtime:", end_time - start_time, "seconds")